# ISYE6420 Homework 3

In [ ]:
import numpy as np
from scipy.stats import beta, norm
from scipy.optimize import fsolve

## Question 1c
Calculate HPD 95%
The pdf must evaluate to the same density at each bound (so k is horizontal).

The probability within those bounds must be equal to 1 - alpha.

Beta **PDF:** $f(x | \alpha, \beta) = \frac{x^{\alpha - 1}(1 - x)^{\beta - 1}}{B(\alpha, \beta)}$

In [2]:
guess_lwr, guess_upr = 0.6, 0.95
a, b = 53.5, 11.5
def credible_interval(x, a, b, dist=beta):
    alpha = 0.05
    lwr, upr = x

    cond_1 = dist.pdf(upr, a, b) - dist.pdf(lwr, a, b)
    cond_2 = (
        dist.cdf(upr, a, b) - dist.cdf(lwr, a, b) - (1 - alpha)       
    )
    return cond_1, cond_2

lower, upper = fsolve(credible_interval, (guess_lwr, guess_upr), args=(a, b))
print(f"HDI Credible Interval: [{lower:.2f}, {upper:.2f}]")
print(
    "Probability within set: ", round(beta.cdf(upper, a, b) - beta.cdf(lower, a, b), 2)
    )
print("Width of interval: ", round(upper - lower, 2))


HDI Credible Interval: [0.73, 0.91]
Probability within set:  0.95
Width of interval:  0.18


## Question 1d
$$
p_0 = \int_{0.8}^{\inf} \pi(\theta|x)d\theta
$$
$$
p_1 = \int_{\inf}^{0.8} \pi(\theta|x)d\theta
$$
Use the CDF of the Beta function to evaluate the two hypotheses and their associated integrals, producing:

In [3]:
p = 0.8
a = 53.5
b = 11.5

prob_left = beta.cdf(p, a, b)
prob_right = 1 - prob_left
print(f"Probability of observing a value < {p}: {prob_left:.2f}")
print(f"Probability of observing a value >= {p}: {prob_right:.2f}")

Probability of observing a value < 0.8: 0.29
Probability of observing a value >= 0.8: 0.71


## Question 2
Assuming the prior also has a normal distribution:
$$
\pi(\theta|\mu) \sim N(\mu, \tau^2)
$$
$$
f(X|\theta) \sim N(\theta, (0.02)^2)
$$
First we find the marginal likelihood for $x_i$
$$
m_i(x_i|\mu) = \int f_i(x_i,\theta_i)\pi(\theta|\mu)d\theta_i
$$
$$
m(x|\mu) = \prod_{i=1}^n m_i(x_i|\mu)
$$
Since $X_i$ distributions are all independent and identically distributed, the variances can simply be added:
$$
X_i \sim_{iid} N(\mu, \sigma^2+\tau^2)
$$
Hence:
$$
m(x|\mu) = \prod_{i=1}^n\frac{1}{\sqrt{2\pi(\sigma^2+\tau^2)}}e^{\frac{-(x_i-\mu)^2}{2(\sigma^2+\tau^2)}}
$$

In [4]:
readings = [3.29, 3.31, 3.35, 3.34, 3.28]
# Mean
mean_reading = np.mean(readings)
print(f"Mean of readings: {mean_reading:.3f}")
# Sample Variance
variance_reading = np.var(readings, ddof=1)
print(f"Variance of readings: {variance_reading:.4f}")
# Standard Deviation
std_dev_reading = np.sqrt(variance_reading)
print(f"Standard Deviation of readings: {std_dev_reading:.4f}")

Mean of readings: 3.314
Variance of readings: 0.0009
Standard Deviation of readings: 0.0305


Using the MLII method, we can say the best estimate for $mu$ is the sample mean $\bar{X}$. Similarly the best estimate for variance is the sample variance, $s^2 =\sigma^2+\tau^2$.
The sample mean can be calculated:
$$
\bar{X} = \frac{3.29+3.31+3.35+3.34+3.28}{5} = 3.314
$$
$$
s^2 = \frac{\sum(x_i-\bar{X})^2}{n-1}=0.0009
$$
$$
\mu_{post}= \frac{\sigma^2x+\tau^2\mu}{\tau^2+\sigma^2} = \hat{B}x + (1-\hat{B})\mu
$$
$$
\tau_{post}^2 = \hat{B}\tau^2
$$


In [5]:
sigma = 0.02
s = std_dev_reading
n = len(readings)
B_hat = sigma**2 / s**2
print(f"Estimated weighting factor B: {B_hat:.4f}")

Estimated weighting factor B: 0.4301


To find $\theta_6$ distribution given $x_6 = 3.30$:

In [6]:
def posterior_mean(B, x_bar, new_x):
    return B*x_bar + (1-B)*new_x

new_x = 3.30
post_mean = posterior_mean(B_hat, mean_reading, new_x)
print(f"Posterior mean for new reading: {post_mean:.4f}")

def posterior_variance(B, sigma):
    return (1-B) * sigma**2
post_var = posterior_variance(B_hat, sigma)
print(f"Posterior variance for new reading: {post_var:.6f}")

Posterior mean for new reading: 3.3060
Posterior variance for new reading: 0.000228


Finding the 95% HDI credible set:

In [7]:
guess_lwr, guess_upr = 3.28, 3.33
a, b = post_mean, post_var**0.5
def credible_interval(x, a, b, dist=norm, alpha = 0.05):
    lwr, upr = x
    cond_1 = dist.pdf(upr, a, b) - dist.pdf(lwr, a, b)
    cond_2 = (
        dist.cdf(upr, a, b) - dist.cdf(lwr, a, b) - (1 - alpha)       
    )
    return cond_1, cond_2

lower, upper = fsolve(credible_interval, (guess_lwr, guess_upr), args=(a, b, norm))
print(f"HDI Credible Interval: [{lower:.2f}, {upper:.2f}]")
print(
    "Probability within set: ", round(norm.cdf(upper, a, b) - norm.cdf(lower, a, b), 2)
    )
print("Width of interval: ", round(upper - lower, 2))

HDI Credible Interval: [3.28, 3.34]
Probability within set:  0.95
Width of interval:  0.06


# Question 3

Parasol mushroom distribution $\sim N(\mu, 9)$

Likelihood: $f(x|\mu=17,n=10) \sim N(17, \frac{\sigma}{10})$

Prior: $\pi(x) \sim N(18, \sigma^2)$

As we are testing for a precise null in a continuous distribution, we need to use a point mass.
$$
\pi(\mu) = \pi_0\delta_{18} + \pi_1\xi(\mu)=0.8\delta_{18} + 0.2\xi(\mu)
$$
where $\pi_0 + \pi_1 = 1$.

The spread distribution can be described as: $ξ(μ) = \frac{1}{\sqrt{2π·12}} e^{-\frac{(μ−18)^2}{2·12}}$.
From the known normal distribution pdf, we can infer that the spread variation is 12 and the spread mean is 18.

In [8]:
measured_mu = 17
n=10
known_var = 9

# Test hypotheses for mu
h1_mu = 18

# Spread distribution for mu under H1
spread_var = 12
spread_mu = 18

measured_likelihood = norm.pdf(measured_mu, loc=measured_mu, scale=np.sqrt(known_var/n))

def spread_dist(mu, spread_var):
    return norm.pdf(mu, loc=18, scale=np.sqrt(spread_var))

def likelihood_func(x, mu, known_var, n):
    return norm.pdf(x, loc=mu, scale=np.sqrt(known_var/n))

def marginal_likelihood(x, var):
    return norm.pdf(x, loc=18, scale=np.sqrt(var))

marginal_var = known_var/n + spread_var
bayes_factor_10 = marginal_likelihood(measured_mu, marginal_var) / likelihood_func(measured_mu, h1_mu, known_var, n)
bayes_factor_01 = 1 / bayes_factor_10
print(f"Bayes Factor in favour of H1: {bayes_factor_10:.3f}")
print(f"Bayes Factor in favour of H0: {bayes_factor_01:.3f}")

Bayes Factor in favour of H1: 0.443
Bayes Factor in favour of H0: 2.258


In [9]:
pi_0 = 0.8

def posterior_prob(bf, pi_0):
    return (1+((1-pi_0)/pi_0) *bf)**(-1)

print(f"Posterior probability of H0: {posterior_prob(bayes_factor_10, pi_0):.3f}")
print(f"Posterior probability of H1: {1 - posterior_prob(bayes_factor_10, pi_0):.3f}")
# Taking the log10 of the Bayes factor for better interpretability
log_bayes_factor_01 = np.log10(bayes_factor_01)
print(f"Log10 of Bayes Factor in favour of H0: {log_bayes_factor_01:.3f}")
log_bayes_factor_10 = np.log10(bayes_factor_10)
print(f"Log10 of Bayes Factor in favour of H1: {log_bayes_factor_10:.3f}")


Posterior probability of H0: 0.900
Posterior probability of H1: 0.100
Log10 of Bayes Factor in favour of H0: 0.354
Log10 of Bayes Factor in favour of H1: -0.354
